# dataloading

In [2]:
# Run this under the directory that contains TorchSpatial, not under TorchSpatial itself
# This file is classification only

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam

from sklearn.model_selection import train_test_split

from TorchSpatial.trainer import train, train_ssi_debias
from TorchSpatial.tester import test
from TorchSpatial.modules.encoder_selector import get_loc_encoder
from TorchSpatial.modules.models import ThreeLayerMLP
import TorchSpatial.utils.datasets as data_import
import TorchSpatial.utils.eval_helper as eval_helper
from TorchSpatial.utils.losses import embedding_loss
from TorchSpatial.modules.models import LocationEncoder

from gbsloss import SSIPartitioner, BinaryPerformanceTransformer, SSILoss, SRIPartitioner, SoftHistogramPerformanceTransformer, SRILoss

from pathlib import Path
import numpy as np
import pandas as pd

import torch
import numpy as np

import json

import warnings

What "loss" is "[epoch 1, batch   450] loss: 0.690" when debiasing? Certainly not the embedding loss, but why do the two match in magnitude?

In [3]:
all_data = data_import.load_dataset(params = {"dataset": "birdsnap", "meta_type": "ebird_meta", "regress_dataset": [], "rand_sample_weight": 0.3},
    eval_split = 'test',
    train_remove_invalid = True,
    eval_remove_invalid = True,
    load_cnn_predictions=True,
    load_cnn_features=True,
    load_cnn_features_train=True)

Loading birdsnap_with_loc_2019.json - train
   using meta data: ebird_meta
	 46386 total entries
	 43426 entries with images
	 42490 entries with meta data
Loading birdsnap_with_loc_2019.json - test
   using meta data: ebird_meta
	 2443 total entries
	 2262 entries with images
	 2217 entries with meta data


In [5]:
all_data.keys()

dict_keys(['train_locs', 'val_locs', 'train_preds', 'train_classes', 'train_users', 'train_dates', 'train_inds', 'train_imgs', 'val_classes', 'val_users', 'val_dates', 'val_inds', 'val_imgs', 'class_of_interest', 'classes', 'num_classes', 'val_preds', 'val_feats', 'train_feats', 'val_split'])

In [13]:
all_data["val_users"]

array([3246, 3102, 1219, ..., 3236, 4661, 3314], shape=(2217,))

In [17]:
len(set(all_data["train_users"]).union(set(all_data["val_users"])))

6159

In [16]:
len(set(all_data["val_users"]))

1682

In [19]:
len(set(all_data["train_users"]))

5763

In [20]:
1682 + 5763 == 6159

False

If user_embedding is to be made, it is to only use the training data, which is the correct implementation. However, what happens if the test data has uses that are not present in the training data, since there are 6159 unique users combines among whom only 5763 are in the training set? 

The classes are always 500, regardless of the training set really has at least an image for all 500 species or not. Thus, the user embedding should at least be the amount of users available up till that point, maybe 300k, to be comprehensive. But that way the user embedding would be too big and impractical to use. 

dict_keys(['train_locs', 'val_locs', 'train_preds', 'train_classes', 'train_users', 'train_dates', 'train_inds', 'train_imgs', 'val_classes', 'val_users', 'val_dates', 'val_inds', 'val_imgs', 'class_of_interest', 'classes', 'num_classes', 'val_preds', 'val_feats', 'train_feats', 'val_split'])

In [7]:
len(all_data["val_users"])

2217